# Guía paso a paso — Actividad nodo sensor (3 horas)

Este cuaderno documenta **comandos**, **parámetros** y **configuración** para la prueba oficial: **360 lecturas cada 30 segundos** (3 h), con **CounterFit** + **SQLite** (`lecturas`), según la propuesta del curso.

> **Nota:** `counterfit` debe ejecutarse en una **terminal externa** (servidor que no termina). Los bloques `powershell` son para **copiar y pegar en PowerShell**; el bloque final de verificación sí puede ejecutarse en Jupyter si el kernel usa el mismo `.venv` y el `.db` ya existe.

## 1. Requisitos previos

| Requisito | Detalle |
|-----------|---------|
| Python | **3.11** o **3.12** (CounterFit suele fallar en 3.13) |
| Proyecto | Carpeta `Nodo Sensor` con `nodo_sensor.py`, `requirements.txt`, `export_lecturas_csv.py` |
| Tiempo | **3 h** continuas; PC sin suspensión; `counterfit` y el script activos |

Ruta de ejemplo en este equipo (ajusta si la tuya es distinta):

`c:\Users\ACER NITRO\Downloads\Nodo Sensor`

## 2. Crear entorno virtual e instalar dependencias

Abre **PowerShell**, ve a la carpeta del proyecto y ejecuta:

Copiar y pegar en **PowerShell** (no ejecutar como Python):

```powershell
cd "c:\Users\ACER NITRO\Downloads\Nodo Sensor"

# Crear venv con Python 3.11 (launcher 'py')
py -3.11 -m venv .venv

# Activar venv
.\.venv\Scripts\Activate.ps1

# Comprobar versión (debe mostrar 3.11.x)
python --version

# Instalar paquetes (incluye pin Werkzeug 2.x para CounterFit)
pip install -r requirements.txt
```

### Si PowerShell bloquea `Activate.ps1`

Una vez, en PowerShell **como administrador**:

```powershell
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser
```

Cierra y vuelve a abrir PowerShell, activa el venv de nuevo.

## 3. Configurar CounterFit (interfaz web)

### 3.1 Arrancar el simulador

En **PowerShell** (con `.venv` activado), **terminal 1**:

```powershell
cd "c:\Users\ACER NITRO\Downloads\Nodo Sensor"
.\.venv\Scripts\Activate.ps1
counterfit
```

Se abre el navegador en **`http://127.0.0.1:5000`** (puerto por defecto **5000**).

### 3.2 Sensores obligatorios (DHT11 virtual)

En la pestaña **Sensors**:

1. **Humidity** — Unidades: **Percentage** — **Pin: 5** — **Add**.
2. **Temperature** — Unidades: **Celsius** — **Pin: 6** — **Add**.

El programa usa `DHT("11", 5)`: humedad en el pin **5** y temperatura en el pin **6** (siguiente pin).

### 3.3 Valores de prueba

En cada tarjeta de sensor, escribe un **Value** razonable (ej. humedad `55`, temperatura `23`), pulsa **Set**.

Opcional: activa **Random** con Min/Max para simular variación durante las 3 h.

### 3.4 Conexión

Cuando ejecutes `nodo_sensor.py` en otra terminal, la UI de CounterFit debe mostrar **Connected** (o indicador equivalente).

## 4. Parámetros del script `nodo_sensor.py`

| Parámetro | Valor **prueba oficial 3 h** | Descripción |
|-----------|------------------------------|-------------|
| `--interval-sec` | **30** | Segundos entre cada lectura |
| `--samples` | **360** | Total de lecturas (360 × 30 s = 10 800 s = 3 h) |
| `--db` | **nodo_sensor.db** | Archivo SQLite de salida (entregar con el informe) |
| `--host` | **127.0.0.1** | Donde corre CounterFit |
| `--port` | **5000** | Puerto HTTP de CounterFit |
| `--dht-pin` | **5** | Pin de humedad; temperatura debe estar en pin **6** |
| `--simulate` | *(no usar)* | Solo para pruebas sin CounterFit; **no** aplica a la entrega con simulador |

**Cálculo:** `(samples - 1) × interval_sec` ≈ tiempo entre primera y última muestra en esperas; con 360 y 30 s son **359 × 30 = 10 770 s** de esperas entre lecturas, más el tiempo de las lecturas (~3 h).

## 5. Comando de la prueba oficial (3 horas)

**Terminal 2** (nueva ventana de PowerShell), con **CounterFit ya corriendo** en la terminal 1 y sensores pin 5 / 6 creados:

```powershell
cd "c:\Users\ACER NITRO\Downloads\Nodo Sensor"
.\.venv\Scripts\Activate.ps1
python nodo_sensor.py --interval-sec 30 --samples 360 --db nodo_sensor.db
```

No cierres la terminal ni suspendas el PC hasta que termine. Verás una línea por lectura en consola y los datos en `nodo_sensor.db` → tabla **`lecturas`**.

## 6. Prueba corta (~1 minuto) antes de las 3 h

Mismos sensores en CounterFit; solo cambian **muestras** e **intervalo**:

```powershell
python nodo_sensor.py --samples 5 --interval-sec 15 --db test_1min.db
```

5 lecturas, 15 s entre ellas → ~60 s de esperas totales.

**Sin CounterFit** (solo para validar SQLite y consola):

```powershell
python nodo_sensor.py --simulate --samples 5 --interval-sec 15 --db test_1min.db
```

## 7. Verificar que hay 360 filas (después de la prueba larga)

En PowerShell con venv activado:

```powershell
python -c "import sqlite3; c=sqlite3.connect('nodo_sensor.db'); print('filas:', c.execute('select count(*) from lecturas').fetchone()[0])"
```

Debe imprimir **`filas: 360`**.

## 8. Exportar a CSV (opcional, para gráficos o anexo)

```powershell
python export_lecturas_csv.py --db nodo_sensor.db -o lecturas.csv
```

Parámetros:
- `--db` — ruta al `.db` (por defecto `nodo_sensor.db`)
- `-o` / `--output` — archivo CSV de salida (por defecto `lecturas.csv`)

## 11. Verificación opcional desde Jupyter

Si el kernel de este notebook usa el intérprete **`.venv\Scripts\python.exe`** y el directorio de trabajo es la carpeta **Nodo Sensor** (donde quedó `nodo_sensor.db` tras la corrida), la siguiente celda cuenta filas sin abrir PowerShell.

En VS Code / Cursor: **Seleccionar kernel** → Python del `.venv` del proyecto; si hace falta, abre el `.ipynb` desde la carpeta del proyecto para que `nodo_sensor.db` se resuelva bien.

In [ ]:
# Ejecutar solo DESPUÉS de la corrida larga y con el kernel en la carpeta del proyecto
import sqlite3
from pathlib import Path

db_path = Path("nodo_sensor.db")
if not db_path.exists():
    print("No existe nodo_sensor.db en el directorio actual del kernel.")
    print("En Jupyter: Kernel -> Cambiar directorio de trabajo, o abrir el notebook desde la carpeta Nodo Sensor.")
else:
    con = sqlite3.connect(db_path)
    n = con.execute("SELECT COUNT(*) FROM lecturas").fetchone()[0]
    con.close()
    print(f"Filas en lecturas: {n} (esperado 360 tras la prueba oficial)")

## 9. Resumen checklist día de la prueba

- [ ] `py -3.11` y `.venv` creado; `pip install -r requirements.txt`
- [ ] CounterFit: **Humidity pin 5**, **Temperature pin 6**; valores o Random configurados
- [ ] Terminal 1: `counterfit` en ejecución
- [ ] Terminal 2: comando de **§5** (`--interval-sec 30 --samples 360`)
- [ ] PC al corriente / sin suspensión ~3 h
- [ ] Tras terminar: verificar **360** filas (§7); copiar `nodo_sensor.db` para entrega
- [ ] Evidencias: capturas CounterFit + consola + (opcional) CSV

## 10. Ayuda en línea del script

```powershell
python nodo_sensor.py --help
```